# Неделя 3. LightGBM и калибровка

**Важно:** `Charged Off` — лишь приближение списания, не подтверждённая DPD90. Использование только завершённых кредитов вызывает смещение. Не использовать для реальных решений. Никакие результаты исходного полного Colab не считаются верифицированными после изменений кода.

In [ ]:
# Запускать после загрузки обновлённого проекта на GitHub.
# В Google Colab код автоматически клонирует публичный репозиторий.
import os, sys, subprocess
from pathlib import Path
if not Path('run_pipeline.py').exists():
    repo = Path('/content/credit-scoring-ml')
    if not (repo / 'run_pipeline.py').exists():
        subprocess.run(['git', 'clone', 'https://github.com/ironrobot19/credit-scoring-ml.git', str(repo)], check=True)
    os.chdir(repo)
print('Папка проекта:', Path.cwd())


Validation делится по времени: TUNE для ранней остановки, CALIBRATION для Platt-сигмоиды, POLICY для выбора денежного порога. Ни один шаг не использует Test.

In [ ]:
from src.train import boost_stage
comparison = boost_stage()
import pandas as pd
display(pd.DataFrame(comparison).round(4))


In [ ]:
import joblib, pandas as pd, matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from src.data import split_validation
from src.features import feature_matrix
v = pd.read_pickle('artifacts/valid.pkl')
cal = split_validation(v)['calibration']
b = joblib.load('artifacts/model_bundle.pkl')
raw = b['model'].predict_proba(b['binner'].transform(feature_matrix(cal)))[:,1]
calibrated = b['calibrator'].predict(raw)
plt.figure(figsize=(6,5))
for name, p in [('Raw LightGBM',raw), ('Platt calibrated',calibrated)]:
    observed, forecast=calibration_curve(cal.target,p,n_bins=8,strategy='quantile')
    plt.plot(forecast,observed,marker='o',label=name)
plt.plot([0,1],[0,1],'--',color='gray');plt.legend();plt.title('Calibration segment (training diagnostic)');plt.show()
print('Порог выбран на POLICY-части validation:',b['threshold'])


**Вывод:** Platt-калибратор действительно обучен на отдельных хронологических наблюдениях. `config.yaml` сохраняет финальные настройки.